In [13]:
import pandas as pd

path = "...NYPD\\"

df = pd.read_csv(path + "NYPD_Arrests_Data_Historic.csv", low_memory=False)

df['OFNS_DESC'] = df['OFNS_DESC'].str.title()
df['PD_DESC'] = df['PD_DESC'].str.title()


In [14]:
df['ARREST_DATE'] = pd.to_datetime(
    df['ARREST_DATE'],
    errors='coerce')

df['ARREST_DATE'].min(), df['ARREST_DATE'].max()

df["Year"] = df['ARREST_DATE'].dt.year

Create Year Category

In [15]:
import pandas as pd

# example: make sure Year is numeric
df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# define bins and labels
bins = [2005, 2010, 2015, 2020, 2025]
labels = ['2006-2010', '2011-2015', '2016-2020', '2021-2024']

# create categorical variable
df['Year Category'] = pd.cut(df['Year'], bins=bins, labels=labels, right=True)

# optional: check results
#df[['Year', 'Year Category']].head()

In [16]:
#year = set(df["Year"].tolist() )
#year

Create Drug Group

In [17]:
df["Description"] =  df['PD_DESC'] + ", " +  df['OFNS_DESC'] 

In [18]:
df["Description"] = df["Description"].astype(str)
def drug_category(desc):
    if 'MARIJUANA' in desc.upper():
        return 'Marijuana'
    elif 'CONTROLLED SUBSTANCE' in desc.upper():
        return 'Other Drugs'
    else:
        return 'Other'

df['Drug Group'] = df['Description'].apply(drug_category)

Filter only drugs

In [19]:
print (len(df))
df = df[df["Drug Group"] != 'Other'].reset_index(drop=True)
print (len(df))



5986025
1124818


In [20]:
cols = df.columns
cols

Index(['ARREST_KEY', 'ARREST_DATE', 'PD_CD', 'PD_DESC', 'KY_CD', 'OFNS_DESC',
       'LAW_CODE', 'LAW_CAT_CD', 'ARREST_BORO', 'ARREST_PRECINCT',
       'JURISDICTION_CODE', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'X_COORD_CD',
       'Y_COORD_CD', 'Latitude', 'Longitude', 'Lon_Lat', 'Description', 'Year',
       'Year Category', 'Drug Group'],
      dtype='object')

In [21]:
subset = ['ARREST_DATE', 'PD_CD', 'PD_DESC', 'KY_CD', 'OFNS_DESC',
       'LAW_CODE', 'LAW_CAT_CD', 'ARREST_BORO', 'ARREST_PRECINCT',
       'JURISDICTION_CODE', 'AGE_GROUP', 'PERP_SEX', 'PERP_RACE', 'Latitude', 'Longitude', 'Description', 'Year',
       'Drug Group', 'Year Category']

df = df[subset]

Clean PD_DESC

In [23]:
df['PD_DESC'] = df['PD_DESC'].str.replace('Controlled Substance, Possessi', 'Controlled Substance, Possession')
df['PD_DESC'] = df['PD_DESC'].str.replace('Controlled Substance,Possess.', 'Controlled Substance, Possession')

df['PD_DESC'] = df['PD_DESC'].str.replace('Controlled Substance, Intent T', 'Controlled Substance, Intent To Sell')
df['PD_DESC'] = df['PD_DESC'].str.replace('Controlled Substance,Intent To', 'Controlled Substance, Intent To Sell')



pd = sorted(list(set(df["PD_DESC"].tolist() )))
pd

['Controlled Substance, Intent To Sell',
 'Controlled Substance, Intent To Sell Sell 3',
 'Controlled Substance, Intent To Sello Sell 5',
 'Controlled Substance, Possession',
 'Controlled Substance, Possession 1',
 'Controlled Substance, Possession 2',
 'Controlled Substance, Possession 3',
 'Controlled Substance, Possession Of Procursers',
 'Controlled Substance, Possessionon 4',
 'Controlled Substance, Possessionon 5',
 'Controlled Substance, Possessionon 7',
 'Controlled Substance, Sale 4',
 'Controlled Substance, Sale 5',
 'Controlled Substance,Sale 1',
 'Controlled Substance,Sale 2',
 'Controlled Substance,Sale 3',
 'Fac. Sexual Offense W/Controlled Substance',
 'Marijuana, Possession',
 'Marijuana, Possession 1, 2 & 3',
 'Marijuana, Possession 4 & 5',
 'Marijuana, Sale 1, 2 & 3',
 'Marijuana, Sale 4 & 5',
 'Unlawful Sale Synthetic Marijuana']

In [24]:
df["Count"] = 1
df.to_csv(path + "NYPD Arrests Data Historic Drugs.csv", index=False)

In [110]:
yc = set(df["Year Category"].tolist() )
yc

{'2006-2010', '2011-2015', '2016-2020', '2021-2024'}

### Spatial Join

In [72]:
import pandas as pd
import geopandas as gpd

# -----------------------------
# 1. Load Community District shapefile
# -----------------------------

pathshape = "C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\nycd_26a\\"

# Load shapefile
cd = gpd.read_file(pathshape + "nycd.shp")

# Make sure CRS is lat/lon
cd = cd.to_crs(epsg=4326)

drug_arrests = df.copy()

# -----------------------------
# Convert arrests to GeoDataFrame using lat/lon
# -----------------------------
drug_gdf = gpd.GeoDataFrame(
    drug_arrests,
    geometry=gpd.points_from_xy(
        drug_arrests["Longitude"],
        drug_arrests["Latitude"]
    ),
    crs="EPSG:4326"
)


# -----------------------------
# Spatial join arrests to community districts
# -----------------------------
drug_joined = gpd.sjoin(
    drug_gdf,
    cd,
    how="inner",
    predicate="within"
)




# -----------------------------
# Count drug arrests by community district
# -----------------------------
drug_counts = (
    drug_joined
    .groupby("BoroCD")
    .size()
    .reset_index(name="Drug Arrest Count")
)

# -----------------------------
# Merge counts back to community district shapefile
# -----------------------------
cd_drug_counts = cd.merge(
    drug_counts,
    on="BoroCD",
    how="left"
)


#Create a loop where year and other categories fields are created

yearlist = sorted(list(set(drug_joined["Year Category"].tolist() )))

print(yearlist)

#year category for all drugs
for i in range(0, len(yearlist)):
    # Count drug arrests by community district

    #drug_joined2 = drug_joined[drug_joined["Year Category"] == yearlist.iloc[i]]
    drug_joined2 = drug_joined[drug_joined["Year Category"] == yearlist[i]]

    fieldname = yearlist[i]
  
    drug_counts = (
        drug_joined2
        .groupby("BoroCD")
        .size()
        .reset_index(name=fieldname)
    )

    drug_countsDic = drug_counts.set_index('BoroCD')[fieldname].to_dict()
    
    cd_drug_counts[fieldname] = cd_drug_counts["BoroCD"].map(drug_countsDic)

#year category for marijuana
print ("before marijuana subset", len(drug_joined))
drug_joined_m = drug_joined[drug_joined["Drug Group"] == "Marijuana"]
print ("after marijuana subset", len(drug_joined_m))

for i in range(0, len(yearlist)):
    # Count drug arrests by community district

    #drug_joined2 = drug_joined[drug_joined["Year Category"] == yearlist.iloc[i]]
    drug_joined2 = drug_joined_m[drug_joined_m["Year Category"] == yearlist[i]]

    fieldname = "M" + yearlist[i]
  
    drug_counts = (
        drug_joined2
        .groupby("BoroCD")
        .size()
        .reset_index(name=fieldname)
    )

    drug_countsDic = drug_counts.set_index('BoroCD')[fieldname].to_dict()
    
    cd_drug_counts[fieldname] = cd_drug_counts["BoroCD"].map(drug_countsDic)
    cd_drug_counts[fieldname] = cd_drug_counts[fieldname].fillna(0).astype(int)

#year category for other drugs
print ("before other drugs subset", len(drug_joined))
drug_joined_m = drug_joined[drug_joined["Drug Group"] == "Other Drugs"]
print ("after other drugs subset", len(drug_joined_m))

for i in range(0, len(yearlist)):
    # Count drug arrests by community district

    #drug_joined2 = drug_joined[drug_joined["Year Category"] == yearlist.iloc[i]]
    drug_joined2 = drug_joined_m[drug_joined_m["Year Category"] == yearlist[i]]

    fieldname = "O" + yearlist[i]
  
    drug_counts = (
        drug_joined2
        .groupby("BoroCD")
        .size()
        .reset_index(name=fieldname)
    )

    drug_countsDic = drug_counts.set_index('BoroCD')[fieldname].to_dict()
    
    cd_drug_counts[fieldname] = cd_drug_counts["BoroCD"].map(drug_countsDic)
    cd_drug_counts[fieldname] = cd_drug_counts[fieldname].fillna(0).astype(int)


# Fill districts with no arrests as 0
cd_drug_counts["Drug Arrest Count"] = cd_drug_counts["Drug Arrest Count"].fillna(0).astype(int)

for i in range(0, len(yearlist)):
    cd_drug_counts[yearlist[i]] = cd_drug_counts[yearlist[i]].fillna(0).astype(int)


# Optional: GeoJSON is often easier for Power BI / web maps
#cd_drug_counts.to_file("community_district_drug_arrests.geojson", driver="GeoJSON")


['2006-2010', '2011-2015', '2016-2020', '2021-2024']
before marijuana subset 1122692
after marijuana subset 512003
before other drugs subset 1122692
after other drugs subset 610689


In [64]:
drug_joined["Drug Group"]

0          Other Drugs
1          Other Drugs
2          Other Drugs
3          Other Drugs
4          Other Drugs
              ...     
1124813    Other Drugs
1124814    Other Drugs
1124815      Marijuana
1124816      Marijuana
1124817    Other Drugs
Name: Drug Group, Length: 1122692, dtype: object

In [52]:
drug_counts

,BoroCD,DA_2021-2024
0,101,463
1,102,971
2,103,1092
3,104,2984
4,105,1484
...,...,...
65,483,2
66,501,2365
67,502,317
68,503,317


In [73]:
cd_drug_counts.head(3)

,BoroCD,Shape_Leng,Shape_Area,geometry,Drug Arrest Count,2006-2010,2011-2015,2016-2020,2021-2024,M2006-2010,M2011-2015,M2016-2020,M2021-2024,O2006-2010,O2011-2015,O2016-2020,O2021-2024
0,410,105822.377310,1.720774e+08,"MULTIPOLYGON (((-73.85722 40.65028, -73.85902 ...",6924,3217,2327,1089,291,1821,1243,375,1,1396,1084,714,290
1,480,47338.739795,3.277756e+07,"POLYGON ((-73.86272 40.76667, -73.86281 40.766...",62,27,21,10,4,15,11,8,0,12,10,2,4
2,483,106865.598387,1.919980e+08,"MULTIPOLYGON (((-73.74694 40.63755, -73.74694 ...",9,3,2,2,2,0,1,1,0,3,1,1,2


Add Community District Name

In [74]:
pathfdny = "C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\FDNY\\"

dp = pd.read_csv(pathfdny + "New_York_City_Population_By_Community_Districts_20260414.csv")
dp["CD Number"] = dp["CD Number"].astype(str)
dp["CD Number"] = dp["CD Number"].str.split(".").str[0]
dp = dp[dp["CD Number"] != 'nan']

dp['Borough2'] = dp['Borough']

dp['Borough2'] = dp['Borough2'].str.replace('Bronx', '2')
dp['Borough2'] = dp['Borough2'].str.replace('Manhattan', '1')
dp['Borough2'] = dp['Borough2'].str.replace('Brooklyn', '3')
dp['Borough2'] = dp['Borough2'].str.replace('Queens', '4')
dp['Borough2'] = dp['Borough2'].str.replace('Staten Island', '5')

dp['boro_cd'] = dp['Borough2'].astype(str) + dp['CD Number'].astype(str).str.zfill(2)

dp['2010 Population'] = dp['2010 Population'].str.replace(',', '')

dp['2010 Population'] = dp['2010 Population'].astype(int)

cd_drug_counts["BoroCD"] = cd_drug_counts["BoroCD"].astype(str)

In [75]:
dpDic = dp.set_index('boro_cd')['CD Name'].to_dict()
cd_drug_counts["Community District Name"] = cd_drug_counts["BoroCD"].map(dpDic)


In [76]:
# -----------------------------
# Export to new shapefile 
# -----------------------------
cd_drug_counts = cd_drug_counts.rename(columns={'Drug Arrest Count': 'DArrests', 'Community District Name': 'CDName'})
cd_drug_counts.to_file(path + "community_district_drug_arrests.shp")

In [77]:
#cd_drug_counts

In [84]:
path

'C:\\Users\\MehriD01\\OneDrive - New York City Housing Authority\\Documents\\NYPD\\'

In [80]:
drug_counts.sort_values(by = 'Drug Arrest Count', ascending=False).reset_index(drop=True)[:50]



,BoroCD,Drug Arrest Count
0,111,53472
1,305,48279
2,112,48119
3,204,46942
4,303,43453
5,201,42920
6,110,42615
7,205,42070
8,412,39797
9,207,37646


In [67]:
cd.head(2)

,BoroCD,Shape_Leng,Shape_Area,geometry
0,410,105822.377310,1.720774e+08,"MULTIPOLYGON (((-73.85722 40.65028, -73.85902 ..."
1,480,47338.739795,3.277756e+07,"POLYGON ((-73.86272 40.76667, -73.86281 40.766..."


In [69]:
df.head(2)

,ARREST_KEY,ARREST_DATE,PD_CD,PD_DESC,KY_CD,OFNS_DESC,LAW_CODE,LAW_CAT_CD,ARREST_BORO,ARREST_PRECINCT,...,PERP_SEX,PERP_RACE,X_COORD_CD,Y_COORD_CD,Latitude,Longitude,Lon_Lat,Description,Year,Drug Group
0,298692639,2024-12-31,511.0,"Controlled Substance, Possessi",235.0,Dangerous Drugs,PL 2200300,M,S,122,...,F,WHITE,955969.0,160925.0,40.608333,-74.101854,POINT (-74.1018540457448 40.608333140421834),"Controlled Substance, Possessi, Dangerous Drugs",2024,Other Drugs
1,298702123,2024-12-31,507.0,"Controlled Substance, Possessi",117.0,Dangerous Drugs,PL 2200605,F,K,83,...,M,WHITE HISPANIC,1008707.0,191689.0,40.692785,-73.911807,POINT (-73.911806532832 40.6927849648184),"Controlled Substance, Possessi, Dangerous Drugs",2024,Other Drugs
